# FreshRoute — Full Project on Colab (CE-1 + CE-2)
Food Bank Distribution Optimizer (India): mandi glut data → cleaning → next-day forecast → dispatch list.

**Run top to bottom.** You need one file on your laptop: `apmc_arrivals_prices.csv` (81 MB, re-download steps in `data/SOURCES.md`). The notebook will ask you to upload it.

In [ ]:
# 0 — Get the repo + libraries
import os
if not os.path.isdir('/content/freshroute'):
    !git clone https://github.com/ParasRana1729/freshroute /content/freshroute
%cd /content/freshroute
!pip install -q xgboost
print('repo ready:', os.getcwd())

In [ ]:
# 1 — Provide the raw data (only step needing your laptop)
import os
RAW = 'data/raw/apmc_arrivals_prices.csv'
if not os.path.exists(RAW):
    print('Upload apmc_arrivals_prices.csv when prompted (81 MB, takes a minute)')
    from google.colab import files
    up = files.upload()
    # move whatever was uploaded to the raw path
    name = list(up.keys())[0]
    os.makedirs('data/raw', exist_ok=True)
    os.replace(name, RAW)
import pandas as pd
df0 = pd.read_csv(RAW, low_memory=False)
print('raw shape:', df0.shape)
df0.head(3)

In [ ]:
# 2 — CE-1: clean + engineer + label (takes ~2-4 min)
!python scripts/clean_dataset.py

In [ ]:
# 3 — CE-1 outputs check
import json, pandas as pd
rep = json.load(open('data/processed/cleaning_report.json'))
print('cleaned:', rep['cleaned_shape'], '| encoded:', rep['encoded_shape'])
print('priority:', rep['target_distribution'])
clean = pd.read_csv('data/processed/freshroute_foodbank_cleaned.csv', nrows=5)
clean[['date','state_name','market_center_name','commodity_name','arrival_tonnes','price_per_kg','surplus_S','redistribution_priority']]

In [ ]:
# 4 — CE-1 figures
from IPython.display import Image, display
for f in ['01_priority_dist','04_top_localities','05_need_vs_supply','07_state_high_rate']:
    display(Image(f'reports/figures/{f}.png', width=650))

In [ ]:
# 5 — CE-2: next-day forecast, LogReg vs RF vs XGBoost (takes ~5-10 min)
!python scripts/train_model.py

In [ ]:
# 6 — CE-2 results + dispatch demo
import json, pandas as pd
m = json.load(open('data/processed/model_metrics.json'))
print('WINNER:', m['winner'], '| test window:', m['test_target_dates'])
pd.DataFrame(m['results']).T
disp = pd.read_csv('data/processed/dispatch_list_demo.csv')
print('dispatch rows:', len(disp))
disp.head(10)

In [ ]:
# 7 — CE-2 figures
from IPython.display import Image, display
for f in ['10_confusion_XGBoost','11_feature_importance']:
    display(Image(f'reports/figures/{f}.png', width=650))